In this document our primary objectives include:

* Comprehending the issue. We will examine each variable, delving into their significance and relevance to the problem at hand.
* Single-variable analysis. Our primary focus will be on the dependent variable manufacturer's suggested retail price (msrp), as we aim to gain deeper insights into it.
* Multi-variable exploration. We will investigate the relationship between the dependent and independent variables.
* Preliminary data cleansing. This will involve addressing missing information, outliers, and categorical variables in the dataset.


# PART I : DATA EXPLORATION WITH PYTHON



In [ ]:
#import some necessary librairies
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns 
import matplotlib.pyplot as plt # Matlab-style plotting
from scipy import stats
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler

In [ ]:
#import file
df=pd.read_csv("../input/cardataset/data.csv")
#set number format
pd.options.display.float_format = '{:.2f}'.format #Limiting floats output to 2 decimal points

In [ ]:
#check DATA columns
df.columns

In [ ]:
#Uniform format
df.columns = df.columns.str.lower().str.replace(' ', '_')
string_columns = list(df.dtypes[df.dtypes == 'object'].index)
for col in string_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')
    
df = df.rename(columns={'msrp': 'price'})

In [ ]:
#check column name
df.head().T

To understand our data, we need to take several steps to organize and analyze our data effectively. The first step is to create an Excel spreadsheet that will help us organize the variables we need to consider. We will use this spreadsheet to help us keep track of each variable and its properties.

After we have created the spreadsheet, we need to fill in two columns: 'Type' and 'Segment.' The 'Type' column helps us identify whether a variable is numerical or categorical, which is important because we need to use different statistical methods to analyze each type. The 'Segment' column helps us categorize variables according to whether they relate to the physical characteristics of the car , properties of the car (horse power), or how many cylinders does the car have (number of cylinders).

Next, we need to fill in the 'Expectation' column for each variable. This helps us develop some sense about the variables that will be most important in predicting car price. To do this, we need to consider whether we would care about the variable when buying a car, how important it is to us, and whether this information is already described in any other variable.

Once we have filled in the 'Expectation' column, we can filter the spreadsheet to focus on the variables with 'High' 'Expectation.' These are the variables that we believe are the most important in predicting car price. We can then create scatter plots to understand the relationship between these variables and the car price. By examining the scatter plots and other relevant data, we can fill in the 'Conclusion' column to reflect our conclusions about the importance of each variable in predicting car price.

By following these steps, we can develop a more accurate car price prediction model and gain a better understanding of the variables that impact car price.

![](https://images.ctfassets.net/2sam6k0rncvg/NqdULBunZY6hmupJnpRJC/d5692087382b9d2baecdc6f16626ce12/top-car-brands-in-india.png?fm=webp&w=1200&q=75)

After completing the analysis, I have arrived at the conclusion that the following variables may have a significant impact on this problem:

* Engine HorsePower.
* YearBuilt.
* Total Cylinders

**Target Variable**

We need to focus on predicting the manufacturer's suggested retail price (MSRP). So, let's start by analyzing this variable.

In [ ]:
sns.distplot(df['price'] , fit=norm);

# Get the fitted parameters used by the function
(mu, sigma) = norm.fit(df['price'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))
#check skewness of the data
print("Skewness: %f" % df['price'].skew())
print("Kurtosis: %f" % df['price'].kurt())

#Now plot the distribution
plt.legend(['Normal dist. ($\mu=$ {:.2f} and $\sigma=$ {:.2f} )'.format(mu, sigma)],
            loc='best')
plt.ylabel('Frequency')
plt.title('SalePrice distribution')

#Get also the QQ-plot
fig = plt.figure()
res = stats.probplot(df['price'], plot=plt)
plt.show()

The target variable exhibits a pronounced right skew. Since (linear) models perform better with normally distributed data, we need to apply a transformation, such as a log transformation, to make the distribution more normal. By implementing the log transform, we can address the skewness and improve the performance of our linear models.

**Log-transformation of the target variable**

In [ ]:
#trasfrom the data to log foramt and then check skewness of the data
#We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
df['log_price'] = np.log1p(df['price'])

#Check the new distribution 
sns.distplot(df['log_price'] , fit=norm);

# Get the fitted parameters used by the function
(mu, sigma) = norm.fit(df['log_price'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))
print("Skewness: %f" % df['log_price'].skew())
print("Kurtosis: %f" % df['log_price'].kurt())

#Now plot the distribution
plt.legend(['Normal dist. ($\mu=$ {:.2f} and $\sigma=$ {:.2f} )'.format(mu, sigma)],
            loc='best')
plt.ylabel('Frequency')
plt.title('SalePrice distribution')

#Get also the QQ-plot
fig = plt.figure()
res = stats.probplot(df['log_price'], plot=plt)
plt.show()

The data now appears to have a more normal distribution, as the skewness has been effectively addressed and corrected.

Let's check out other feature

In [ ]:
df.head()

In [ ]:
# Set the variable and data for the scatter plot
engine_col = 'engine_hp'
engine_data = pd.concat([df['price'], df[engine_col]], axis=1)
# Create the scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x=engine_data[engine_col], y=engine_data['price'])
ax.set_ylim([0, 800000])
ax.set_title("Scatter plot of car popularity and price")
ax.set_xlabel("Engine Horsepower (rpm)")
ax.set_ylabel("Price ($)")

# Show the plot
plt.show()

In [ ]:
# Set the variable and data for the scatter plot
popularity_col = 'popularity'
popularity_price_data = pd.concat([df['price'], df[popularity_col]], axis=1)
# Create the scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x=popularity_price_data[popularity_col], y=popularity_price_data['price'])
ax.set_ylim([0, 800000])
ax.set_title("Scatter plot of car popularity and price")
ax.set_xlabel("Popularity Score")
ax.set_ylabel("Price ($)")

# Show the plot
plt.show()

In [ ]:
# Set the variable and data for the box plot
engine_cylinders_col = 'engine_cylinders'
engine_cylinders_price_data = pd.concat([df['price'], df[engine_cylinders_col]], axis=1)


# Create the box plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(x=engine_cylinders_col, y='price', data=engine_cylinders_price_data, ax=ax)
ax.set_title('Box plot of engine cylinder and price')
ax.set_xlabel('Number of Engine Cylinders')
ax.set_ylabel('Price ($)')
plt.xticks(rotation=45)

# Show the plot
plt.show()

In [ ]:
# Select the top car makes by frequency
make_col = 'make'
top_makes = df[make_col].value_counts().head(12).index.tolist()


# Select the top 5 car makes by frequency
top_makes = df['make'].value_counts().head(12).index.tolist()

# Create a new DataFrame that only includes the top makes
top_make_data = df[df[make_col].isin(top_makes)]

# Create the box plot with the top makes
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(x=make_col, y='price', data=top_make_data, ax=ax)
ax.set_title('Box plot of top car brands and price')
ax.set_xlabel('Car Brand')
ax.set_ylabel('Price ($)')
plt.xticks(rotation=45)

# Show the plot
plt.show()

In [ ]:
# Set the variable and data for the box plot
year_col = 'year'
year_price_data = pd.concat([df['price'], df[year_col]], axis=1)

# Create the box plot
fig, ax = plt.subplots(figsize=(16, 8))
sns.boxplot(x=year_col, y="price", data=year_price_data, ax=ax)
ax.set_ylim([0, 800000])
ax.set_title("Box plot of car year and price")
ax.set_xlabel("Year")
ax.set_ylabel("Price ($)")
plt.xticks(rotation=90)

# Show the plot
plt.show()

In [ ]:
#check correlation
plt.figure(figsize=(7,6))
correlation = df.corr()
sns.heatmap(correlation,cmap="coolwarm",annot=True)
correlation

In [ ]:
#scatterplot of price
sns.set()
cols = ['year', 'engine_hp', 'engine_cylinders', 'number_of_doors', 'price',]
sns.pairplot(df[cols], height = 2.5)
plt.show();

In [ ]:
#scatterplot of log_price
sns.set()
cols = ['year', 'engine_hp', 'engine_cylinders', 'number_of_doors', 'log_price',]
sns.pairplot(df[cols], height = 2.5)
plt.show();

While we are already familiar with some of the key metrics, this comprehensive scatter plot offers a valuable overview of the relationships between variables.

A noteworthy relationship can be observed between 'engine horsepower' and 'engine cylinders'. In the corresponding graph, the data points seem to form a linear pattern. It is quite logical that the majority of these points are situated below this line. This is because the number of engine cylinders can influence the engine horsepower, as having more cylinders generally leads to increased power output


# **PART II : BASIC DATA CLEANSING**

Check Missing Data

In [ ]:
#check missing ratio
data_na = (df.isnull().sum() / len(df)) * 100
data_na = data_na.drop(data_na[data_na == 0].index).sort_values(ascending=False)[:30]
missing_data = pd.DataFrame({'Missing Ratio' :data_na})
missing_data.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))
sns.barplot(x=data_na.index, y=data_na, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set(xlabel='Features', ylabel='Percent of missing values', 
       title='Percent missing data by feature')
ax.grid(True)
plt.show()


Drop Duplicate

In [ ]:
#check duplicate
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

In [ ]:
#drop duplicates
df = df.drop_duplicates()

**Deal with missing values**

GROUPBY METHOD

we use the groupby function along with transform to handle missing data in the DataFrame 'df'. The purpose is to fill missing values with similar values from other records based on the 'model' column, which is assumed to be a meaningful reference for filling the gaps.

* For the 'engine_fuel_type' column, the missing values are filled with the mode (most frequent value) of the 'engine_fuel_type' within the same 'model' group. If the mode is empty, it uses 'None' as the fallback value.
* For the 'number_of_doors' column, the missing values are filled with the mean (average) of the 'number_of_doors' within the same 'model' group.
* For the 'engine_cylinders' column, the missing values are filled with the mean (average) of the 'engine_cylinders' within the same 'model' group.
* For the 'engine_hp' column, the missing values are filled with the mean (average) of the 'engine_hp' within the same 'model' and 'year' groups.

In [ ]:
# fill the null with similiar value in model
df['engine_fuel_type'] = df.groupby('model')['engine_fuel_type'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else None))
df['number_of_doors'] = df.groupby('model')['number_of_doors'].transform(lambda x: x.fillna(x.mean()))
df['engine_cylinders'] = df.groupby('model')['engine_cylinders'].transform(lambda x: x.fillna(x.mean()))
df['engine_hp'] = df.groupby(['model', 'year'])['engine_hp'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
#check for null values
df.isnull().sum()

As we utilize the groupby method, there may still be null values present in our dataset. To address this issue, we can use a rule-based method for imputing these remaining missing values



RULE-BASED IMPUTATION

For filling missing values in the 'engine_cylinders' column of the DataFrame 'df'.

The first part of the code targets records where the 'engine_cylinders' value is missing (null) and the 'model' is either 'rx-7' or 'rx-8'. The code uses the loc function to locate these records and assigns a value of 4 to the 'engine_cylinders' column for these rows. This implies that, based on domain knowledge, both 'rx-7' and 'rx-8' models are assumed to have 4 engine cylinders.

The second part of the code addresses records where the 'engine_cylinders' value is missing (null) and the 'engine_fuel_type' is 'electric'. For these records, the code sets the 'engine_cylinders' value to 0. This is because electric cars do not have traditional internal combustion engines with cylinders.


![](https://i.vimeocdn.com/video/492279142-6e9708c891b1b374767407047b796400953d0771d3105e5ad7ef0924b9d6f5f6-d_640)


In [ ]:
#fill the value with loc
df.loc[((df['engine_cylinders'].isna()) & (df['model'] == 'rx-7')) |
        ((df['engine_cylinders'].isna()) & (df['model'] == 'rx-8')), 
        'engine_cylinders'] = 4

#If the car is an electric car, then it does not have an engine cylinder.
df.loc[(df['engine_cylinders'].isna()) & (df['engine_fuel_type'] == 'electric'), 
       'engine_cylinders'] = 0

For filling missing values in the 'engine_hp' column of the DataFrame 'df'.

We have utilized the information provided on a specific website to determine the engine horsepower values for our analysis.

(https://www.motortrend.com/)

In [ ]:
# fiat
df.loc[(df['engine_hp'].isna()) & (df['model'] == '500e'), 'engine_hp'] = 118
# honda
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'fit_ev'), 'engine_hp'] = 123
# ford
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'freestar'), 'engine_hp'] = 194
# mitsubishi
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'i-miev'), 'engine_hp'] = 66
# nissan
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'leaf'), 'engine_hp'] = 107
# toyota
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'rav4_ev'), 'engine_hp'] = 154
# kia
df.loc[(df['engine_hp'].isna()) & (df['model'] == 'soul_ev'), 'engine_hp'] = 147
# tesla
df.loc[((df['engine_hp'].isna())  & (df['model'] == 'model_s') & (df['year'] == 2014)), 'engine_hp'] = 362
df.loc[((df['engine_hp'].isna())  & (df['model'] == 'model_s') & (df['year'] == 2015)), 'engine_hp'] = 380
df.loc[((df['engine_hp'].isna())  & (df['model'] == 'model_s') & (df['year'] == 2016)), 'engine_hp'] = 315

In [ ]:
df.drop('market_category', axis = 1, inplace = True)

In [ ]:
#check for null values
df.isnull().sum()

After you've explored your data and cleaned it, follow these steps to create a machine learning model for predicting car prices. These steps are simplified to make them more understandable for beginners:

* Feature Engineering: Figure out which pieces of information (like car make, model, year, etc.) are important for predicting car prices. You might also create new information or change existing information to better understand patterns in the data.

* Train-Test Split: Divide your data into two parts: one for training the model (usually 70-90% of the data) and one for testing its performance (the remaining 10-30%).

* Model Selection: Pick a suitable machine learning technique for predicting car prices based on your data. Some popular choices include Linear Regression, Decision Trees, Random Forests, and Neural Networks.

* Model Training: Teach your chosen model using the training data so it can learn the relationship between the important pieces of information and car prices.

* Model Evaluation: Test the trained model on the testing data to see how well it performs. Common ways to measure performance include Mean Absolute Error, Mean Squared Error, and R-squared.

**If you discovered this notebook to be useful or enjoyable, I'd greatly appreciate any upvotes! Your support motivates me to regularly update and improve it. :-)**